# Fundamentos de observabilidad en agentes LLM

Este notebook introduce qué significa observar un agente de lenguaje, por qué es más complejo que observar una API tradicional y qué información debemos capturar para entender su comportamiento.

## Objetivos
- Comprender la diferencia entre observabilidad y depuración en agentes LLM.
- Identificar las señales clave: prompts, respuestas, modelo, tokens, latencia, costo, errores y herramientas usadas.
- Construir una traza simple de ejecución usando Python y `pandas`.

## Temas
- Observabilidad en software tradicional: logs, métricas y trazas.
- Por qué un agente LLM es más difícil de observar.
- Diferencias entre: llamada simple, cadena de pasos, agente con herramientas, agente con memoria y agente multi-paso.
- Qué registrar en cada ejecución.

## 1. Observabilidad tradicional vs observabilidad de agentes LLM

En sistemas tradicionales, observamos principalmente:
- `logs` de eventos
- `métricas` de rendimiento (latencia, tasa de errores)
- `trazas` de ejecución entre servicios.

Para un agente LLM, además de esos datos, necesitamos saber:
- qué prompt se envió
- qué modelo se usó
- cuántos tokens se consumieron
- si el agente llamó a una herramienta
- por qué tomó una decisión en cada paso.

In [11]:
!pip install pandas langchain langchain-openai langchain_classic wikipedia LangSmith


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import wikipedia
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()  # Carga las variables de entorno desde el archivo .env

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model="openai/gpt-oss-120b",
        openai_api_base=os.environ.get("GROQ_BASE_URL"),
        openai_api_key=os.environ.get("GROQ_API_KEY"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

✅ LLM de LangChain configurado.


In [13]:
# Importar herramientas de análisis y simulación
import os
import time
import uuid 
from pprint import pprint

# Intentar importar pandas con mensaje claro si no está instalado
try:
    import pandas as pd
except ImportError:
    raise ImportError(
        'Este notebook usa pandas. Ejecuta `pip install pandas` antes de continuar.'
    )

from langchain_core.messages import HumanMessage

# Función de ayuda para invocar el LLM real

def call_llm(prompt_text):
    if llm is None:
        raise RuntimeError("El LLM no está configurado. Revisa GROQ_API_BASE y GROQ_API_KEY.")
    # Función para normalizar la respuesta del LLM y extraer el texto
    def extract_text(response):
        if hasattr(response, "content"):
            return response.content
        if hasattr(response, "text"):
            return response.text
        if hasattr(response, "generations") and response.generations:
            return response.generations[0][0].text
        return str(response)

    try:
        message = HumanMessage(content=prompt_text)
        if hasattr(llm, "predict_messages"):
            response = llm.predict_messages([message])
        elif hasattr(llm, "generate"):
            response = llm.generate([[message]])
        else:
            response = llm([message])

        text = extract_text(response)
        model_name = getattr(llm, "model_name", getattr(llm, "model", "openai/gpt-oss-120b"))
        prompt_tokens = 0
        completion_tokens = 0
        #si la respuesta tiene información de uso, extraer los tokens
        if hasattr(response, "usage") and response.usage is not None:
            prompt_tokens = getattr(response.usage, "prompt_tokens", 0)
            completion_tokens = getattr(response.usage, "completion_tokens", 0)
        return {
            "text": text,
            "model": model_name,
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "latency": 0.0,
            "cost": 0.0,
            "error": None,
        }
    except Exception as e:
        return {
            "text": "",
            "model": getattr(llm, "model_name", getattr(llm, "model", "openai/gpt-oss-120b")),
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "latency": 0.0,
            "cost": 0.0,
            "error": str(e),
        }


In [ ]:
# clase para registrar trazas de observabilidad
class ObservabilityTrace:
    def __init__(self):
        #atribto de la clase
        self.events = []
        #agrega un evento a la lista
    def record(self, event):
        # Guardar una traza simple con las señales más importantes
        self.events.append(event)
    #convierte lalista en un dtaframe
    def to_dataframe(self):
        return pd.DataFrame(self.events)

    def display(self):
        df = self.to_dataframe()
        display(df)

# Agente simple con distintos modos de ejecución
class MiniAgent:
    def __init__(self, llm):
        self.llm = llm
        self.memory = []
        self.model_name = getattr(llm, "model_name", getattr(llm, "model", "openai/gpt-oss-120b"))


    #metodo que arma cada fila de traza y lo guarda en el trace (diccionario de eventos)
    def _record_step(self, trace, step_type, prompt, output, decision, error=None):
        trace.record({
            'step_id': str(uuid.uuid4()),
            'step_type': step_type,
            'prompt': prompt,
            'output': output,
            'model': self.model_name,
            'decision': decision,
            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
            'error': error,
        })

    def _generate(self, prompt):
        return call_llm(prompt)
    
    # metodo 1: hacer una llamada simple al modelo, sin pasos intermedios
    def simple_call(self, user_question, trace):
        result = self._generate(user_question)
        self._record_step(
            trace,
            step_type='llm-simple',
            prompt=user_question,
            output=result['text'],
            decision='uso modelo directo',
            error=result['error'],
        )
        return result

    # metodo 2: un agente que descompone la pregunta y luego genera la respuesta
    def chain_of_steps(self, user_question, trace):
        # Un agente que transforma la pregunta y luego genera la respuesta
        analysis_prompt = f'Analiza la pregunta para extraer intención: {user_question}'
        analysis = self._generate(analysis_prompt)
        self._record_step(
            trace,
            step_type='llm-analyze',
            prompt=analysis_prompt,
            output=analysis['text'],
            decision='descomponer pregunta',
            error=analysis['error'],
        )

        final_prompt = f'Con base en el análisis previo, responde: {user_question}'
        final = self._generate(final_prompt)
        self._record_step(
            trace,
            step_type='llm-answer',
            prompt=final_prompt,
            output=final['text'],
            decision='respuesta final',
            error=final['error'],
        )
        return final
    # metodo 3: un agente que decide si usar una herramienta o no
    def agent_with_tools(self, user_question, trace):

        decision_prompt = f'¿Necesito una herramienta para contestar: {user_question}?'
        decision = self._generate(decision_prompt)
        self._record_step(
            trace,
            step_type='llm-decision',
            prompt=decision_prompt,
            output=decision['text'],
            decision='evaluar uso de herramienta',
            error=decision['error'],
        )
        # en un se puede usar lagchaing con @tool, pero aquí simulamos una herramienta simple de calculadora
        if 'calcula' in decision['text'].lower() or 'suma' in decision['text'].lower():
            tool_output = str(sum(int(n) for n in user_question.split() if n.isdigit()))
            self._record_step(
                trace,
                step_type='tool',
                prompt='Calculadora local',
                output=tool_output,
                decision='usar calculadora',
            )
            answer = f'El resultado de la operación es {tool_output}.'
            self._record_step(
                trace,
                step_type='llm-answer',
                prompt=f'Responde usando el resultado de la calculadora: {tool_output}',
                output=answer,
                decision='formular respuesta con herramienta',
            )
            return {'text': answer, 'model': self.model_name, 'prompt_tokens': 0, 'completion_tokens': 0, 'latency': 0.05, 'cost': 0.0, 'error': None}

        answer = self._generate(user_question)
        self._record_step(
            trace,
            step_type='llm-answer',
            prompt=user_question,
            output=answer['text'],
            decision='respuesta directa sin herramienta',
            error=answer['error'],
        )
        return answer
    # metodo 4: un agente con memoria que recuerda interacciones anteriores
    def memory_agent(self, user_question, trace):
        # Un agente con memoria que recuerda interacciones anteriores
        history = ' | '.join(self.memory[-2:])
        prompt = f'Historial: {history} Pregunta: {user_question}' if history else user_question
        result = self._generate(prompt)
        self.memory.append(user_question)
        self._record_step(
            trace,
            step_type='llm-memory',
            prompt=prompt,
            output=result['text'],
            decision='usar contexto de memoria',
            error=result['error'],
        )
        return result
    # metodo 5: un agente multi-paso que analiza, decide y responde
    def multi_step_agent(self, user_question, trace):
        
        step1 = self._generate(f'Analiza y resume la pregunta: {user_question}')
        self._record_step(
            trace,
            step_type='llm-analyze',
            prompt=f'Analiza y resume la pregunta: {user_question}',
            output=step1['text'],
            decision='extraer intención',
            error=step1['error'],
        )

        step2 = self._generate(f'¿Deberíamos usar un dato externo para: {user_question}?')
        self._record_step(
            trace,
            step_type='llm-decision',
            prompt=f'¿Deberíamos usar un dato externo para: {user_question}?',
            output=step2['text'],
            decision='evaluar herramienta externa',
            error=step2['error'],
        )

        if 'sí' in step2['text'].lower() or 'necesito' in step2['text'].lower():
            tool_output = '42'
            self._record_step(
                trace,
                step_type='tool',
                prompt='Consulta a base de conocimiento local',
                output=tool_output,
                decision='consulta externa',
            )
            final_text = f'Usé la herramienta externa y el resultado es {tool_output}.'
        else:
            final = self._generate(user_question)
            final_text = final['text']

        self._record_step(
            trace,
            step_type='llm-answer',
            prompt='Genera la respuesta final',
            output=final_text,
            decision='respuesta multi-paso',
            error=None,
        )
        return {'text': final_text, 'model': self.model_name, 'prompt_tokens': 0, 'completion_tokens': 0, 'latency': 0.0, 'cost': 0.0, 'error': None}


In [17]:
# Crear un agente real y registrar varias ejecuciones
if llm is None:
    raise RuntimeError("El LLM no está configurado. Verifica las credenciales y las variables de entorno.")

agent = MiniAgent(llm)

# Ejecución 1: llamada simple a LLM
trace_simple = ObservabilityTrace()
result_simple = agent.simple_call('¿Cuál es la capital de Francia?', trace_simple)
print('=== Respuesta simple ===')
print(result_simple['text'])
print()
trace_simple.display()

# Ejecución 2: cadena de pasos
trace_chain = ObservabilityTrace()
_ = agent.chain_of_steps('Explícame cómo funciona una batería.', trace_chain)
print('=== Traza de cadena de pasos ===')
trace_chain.display()

# Ejecución 3: agente con herramientas
trace_tool = ObservabilityTrace()
_ = agent.agent_with_tools('Calcula 13 + 29.', trace_tool)
print('=== Traza de agente con herramientas ===')
trace_tool.display()

# Ejecución 4: agente con memoria
trace_memory = ObservabilityTrace()
_ = agent.memory_agent('¿Qué fue lo último que pregunté?', trace_memory)
print('=== Traza de agente con memoria ===')
trace_memory.display()

# Ejecución 5: agente multi-paso
trace_multi = ObservabilityTrace()
_ = agent.multi_step_agent('¿Debo usar una herramienta para responder esta pregunta?', trace_multi)
print('=== Traza de agente multi-paso ===')
trace_multi.display()

=== Respuesta simple ===
La capital de Francia es **París**.



,step_id,step_type,prompt,output,model,decision,timestamp,error
0,5e1dfe03-fa8f-4baf-8868-a51c9406ce98,llm-simple,¿Cuál es la capital de Francia?,La capital de Francia es **París**.,openai/gpt-oss-120b,uso modelo directo,2026-11-02 21:35:38,None


=== Traza de cadena de pasos ===


,step_id,step_type,prompt,output,model,decision,timestamp,error
0,1a351a83-86e4-4127-a56c-4a788239c879,llm-analyze,Analiza la pregunta para extraer intención: Ex...,**Análisis de la pregunta “Explícame cómo func...,openai/gpt-oss-120b,descomponer pregunta,2026-11-02 21:35:40,None
1,4654b6ee-306d-4443-89cb-2fc889eae38a,llm-answer,"Con base en el análisis previo, responde: Expl...",### ¿Cómo funciona una batería? \n\nUna bater...,openai/gpt-oss-120b,respuesta final,2026-11-02 21:35:45,None


=== Traza de agente con herramientas ===


,step_id,step_type,prompt,output,model,decision,timestamp,error
0,8b6698e3-035c-467d-a4e7-e0d1d646b189,llm-decision,¿Necesito una herramienta para contestar: Calc...,**Respuesta:** 13 + 29 = **42**.,openai/gpt-oss-120b,evaluar uso de herramienta,2026-11-02 21:35:46,None
1,6125f600-dff3-44f6-964f-0a1cb3593a0f,llm-answer,Calcula 13 + 29.,13 + 29 = 42.,openai/gpt-oss-120b,respuesta directa sin herramienta,2026-11-02 21:35:46,None


=== Traza de agente con memoria ===


,step_id,step_type,prompt,output,model,decision,timestamp,error
0,aff6b266-5c30-4bd8-a723-e7c919a3960b,llm-memory,¿Qué fue lo último que pregunté?,La última cosa que preguntaste fue: **«¿Qué fu...,openai/gpt-oss-120b,usar contexto de memoria,2026-11-02 21:35:47,None


=== Traza de agente multi-paso ===


,step_id,step_type,prompt,output,model,decision,timestamp,error
0,f57a1121-1eda-47ec-9dd6-1018a8017a7a,llm-analyze,Analiza y resume la pregunta: ¿Debo usar una h...,**Análisis de la pregunta**\n\n- **Tipo de pre...,openai/gpt-oss-120b,extraer intención,2026-11-02 21:35:49,None
1,dc2611c5-bcdf-4e09-9d03-ed3520d9c5e1,llm-decision,¿Deberíamos usar un dato externo para: ¿Debo u...,"**Respuesta breve:** \nSí, en la mayoría de l...",openai/gpt-oss-120b,evaluar herramienta externa,2026-11-02 21:35:52,None
2,11c743ae-fabb-49f2-b37b-120c07c0cedc,tool,Consulta a base de conocimiento local,42,openai/gpt-oss-120b,consulta externa,2026-11-02 21:35:52,None
3,066c9cde-e790-4429-b11d-ce7047a60378,llm-answer,Genera la respuesta final,Usé la herramienta externa y el resultado es 42.,openai/gpt-oss-120b,respuesta multi-paso,2026-11-02 21:35:52,None


In [24]:
print('=== Respuesta del modelo (cadena de pasos) ===')
print(trace_chain.to_dataframe()['output'].iloc[-1])
print()
print('=== Traza de cadena de pasos ===')
trace_chain.display()


=== Respuesta del modelo (cadena de pasos) ===
### ¿Qué es una batería y cómo funciona?

Una **bateria** es un dispositivo que almacena energía química y la convierte en energía eléctrica cuando la necesitamos. Está compuesta por una o varias **celdas electroquímicas** conectadas en serie o en paralelo. Cada celda contiene tres componentes esenciales:

| Componente | Función | Ejemplo típico |
|------------|---------|----------------|
| **Ánodo** (electrodo negativo) | Donde ocurre la oxidación (pérdida de electrones). | Zinc (Zn) en una pila alcalina. |
| **Cátodo** (electrodo positivo) | Donde ocurre la reducción (ganancia de electrones). | Dióxido de manganeso (MnO₂) en una pila alcalina. |
| **Electrolito** | Medio que permite el movimiento de iones entre ánodo y cátodo, completando el circuito interno. | Solución de hidróxido de potasio (KOH) en una pila alcalina. |

---

## 1. Principio básico: Reacciones redox

Una batería funciona gracias a **reacciones de óxido‑reducción (redo

,step_id,step_type,prompt,output,model,decision,timestamp,error
0,73bf8e31-5f84-4558-b5b4-ef92d61d4050,llm-analyze,Analiza la pregunta para extraer intención: Ex...,**Análisis de la pregunta “Explícame cómo func...,openai/gpt-oss-120b,descomponer pregunta,2026-11-02 21:37:52,None
1,fca8b281-e7c1-46b5-8351-bd6b28f302fb,llm-answer,"Con base en el análisis previo, responde: Expl...",### ¿Qué es una batería y cómo funciona?\n\nUn...,openai/gpt-oss-120b,respuesta final,2026-11-02 21:37:56,None
